# GRPO experiments

!use the trl environment!

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"  # Use GPU 2

## Load model and datapipe

In [9]:
# load datapipe
import sys
sys.path.append("../spectus")
from utils.data_utils import build_single_datapipe
from utils.tokenizer_utils import build_tokenizer
from train_spectus import get_spectro_config

tokenizer_path = "../tokenizer/tokenizer_mf10M.model"
tokenizer = build_tokenizer(tokenizer_path)
preprocess_args = {
            "restrict_intensities": False,
            "inference_mode": False,
            "max_num_peaks": 300,
            "max_mol_repr_len": 100,
            "max_mz": 1000,
            "mol_repr": "selfies" if tokenizer_path == "selfies_tokenizer" else "smiles",
            "log_base": 1.28,
            "log_shift": 29,
            "max_cumsum": None,
            "tokenizer": tokenizer,
            "do_log_binning": True,
            "linear_bin_decimals": None,
            "output_format": "<mol_repr>",
        }

model_args = {
    "decoder_seq_len": 200,
    "max_mz": 500,
    "separate_encoder_decoder_embeds": True,
    "encoder_layers": 12,
    "encoder_ffn_dim": 4096,
    "encoder_attention_heads": 16,
    "decoder_layers": 12,
    "decoder_ffn_dim": 4096,
    "decoder_attention_heads": 16,
}
if preprocess_args["do_log_binning"]:
    model_args["max_log_id"] = preprocess_args["log_shift"]
else:
    if not preprocess_args.get("linear_bin_decimals", None):
        raise ValueError("linear_bin_decimals must be provided if do_log_binning is False. It's 2 for 100 bins, 3 for 1000 bins, ...")
    model_args["max_log_id"] = 10**preprocess_args["linear_bin_decimals"]



# load datapipe
data_path = "../data/nist/train.jsonl"
datapipe = build_single_datapipe(data_path,
                                 shuffle=True,
                                 buffer_size=10000,
                                 limit=10000,
                                 source_token="<nist>"
                                 )

spectus_spectro_config = get_spectro_config(model_args, tokenizer)
print("Loading model...")
if checkpoint is not None:
    print(f"Loading checkpoint from {checkpoint}")
    model = SpectusForConditionalGeneration.from_pretrained(checkpoint)
else:
    model = SpectusForConditionalGeneration(spectus_spectro_config)





## Quickstart (HF)

In [2]:
# train_grpo.py
from datasets import load_dataset
from trl import GRPOConfig, GRPOTrainer

dataset = load_dataset("trl-lib/tldr", split="train")

# Define the reward function, which rewards completions that are close to 20 characters
def reward_len(completions, **kwargs):
    return [-abs(20 - len(completion)) for completion in completions]

training_args = GRPOConfig(output_dir="Qwen2-0.5B-GRPO", logging_steps=10)
trainer = GRPOTrainer(
    model="Qwen/Qwen2-0.5B-Instruct",
    reward_funcs=reward_len,
    args=training_args,
    train_dataset=dataset,
)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/home/xhajek9/miniconda3/envs/trl/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/home/xhajek9/miniconda3/envs/trl/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/home/xhajek9/miniconda3/envs/trl/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/xhajek9/miniconda3/envs/trl/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, 

model.safetensors:  86%|########5 | 849M/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [4]:
type(dataset)

datasets.arrow_dataset.Dataset

In [ ]:
trainer.train()
